# A3 | 11– Revisar desenho experimental e pilotos

**Pessoa.** Verificar se métodos, métricas, partições e outputs respondem à pergunta sem contaminar o holdout.

**Saída:** checklist metodológico, atas dos **Gates 2** e **3** e registro de interpretação dos pilotos.
**Revisão:** B explica clustering; C explica pipeline, validação e métricas.

In [ ]:
# ----- Setup (não editar) -----
import os, sys
from pathlib import Path

# PKG_ROOT: pasta que contém src/protocolo (onde o pacote foi copiado)
# REPO_ROOT: raiz do repositório (tem .git ou data/); é de onde lemos dados e gravamos saídas
_parents = [Path.cwd(), *Path.cwd().parents]
PKG_ROOT = next((p for p in _parents if (p / "src" / "protocolo").exists()), None)
if PKG_ROOT is None:
    raise FileNotFoundError(
        "Pasta src/protocolo não encontrada acima de " + str(Path.cwd()) + ". "
        "Copie 'src/protocolo/' do pacote para o 'src/' do repositório "
        "(deve existir src/protocolo/verificacoes.py) e rode a célula de novo.")
REPO_ROOT = next((p for p in _parents if (p / ".git").exists() or (p / "data").exists()), PKG_ROOT)
os.chdir(REPO_ROOT); sys.path.insert(0, str(PKG_ROOT))

# Proteção contra outro pacote chamado 'src' (ex.: instalado por pip) ou 'src' antigo em cache no kernel:
# um pacote regular em qualquer ponto do sys.path vence um 'src' sem __init__.py.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]
_init = PKG_ROOT / "src" / "__init__.py"
if not _init.exists():
    _init.touch()
    print("Criado", _init, "(torna o src do projeto um pacote regular)")

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from src.protocolo import verificacoes as v
if getattr(v, "MODULE_VERSION", "0") < "0.8":
    raise RuntimeError(
        "src/protocolo/verificacoes.py está desatualizado (versão " + str(getattr(v, "MODULE_VERSION", "antiga")) + "). "
        "Substitua pelo arquivo do pacote novo e REINICIE o kernel (o Python mantém o módulo antigo em cache).")

# ----- Parâmetros -----
USAR_DEMO = False                       # True = dados sintéticos só para testar o notebook
DATA_PATH = None                        # None = procura Parquet/CSV em data/ ; ou informe o caminho do arquivo
SAMPLE = 0                              # ex.: 200_000 para execução rápida
OUT = "outputs/tables/pessoa"         # saídas (tabelas, atas, decisões), relativas à raiz do repositório
CFG_PATH = next((str(p) for p in [REPO_ROOT / "configs" / "decisoes.yaml", PKG_ROOT / "configs" / "decisoes.yaml"]
                 if p.exists()), "configs/decisoes.yaml")
SUPERVISED_CONFIG = "configs/supervised.yaml"   # config do C (alvo, features, coluna_tempo)
COLUNAS = "minimas"                   # "minimas" (menos memória) | "classificadas" | "todas" (pesado!) | lista de nomes
SUPERVISED_CONFIG = "configs/supervised.yaml"   # config do C (G1–G8)
FEATURES_CLUSTERING = None                      # .txt/.json/.yaml com as features do clustering (B)
LOGS_DIR = "logs"                               # logs de B e C
RESULTS = None                                  # CSV: modelo, fold, pr_auc, roc_auc, brier (pilotos)
print("Raiz do repositório:", REPO_ROOT)
print("Pacote (src/protocolo):", PKG_ROOT)
print("Config de decisões:", CFG_PATH)
cfg = v.load_config(CFG_PATH)
rep = v.Report(OUT, "relatorio_A3")
df = v.prepare(v.carregar_base(DATA_PATH, USAR_DEMO, SAMPLE, COLUNAS, cfg, SUPERVISED_CONFIG))
print(f"Memória do DataFrame: {v.uso_memoria_gb(df):.2f} GB")

## 1. Auditoria do `supervised.yaml` contra as decisões da Pessoa
Alvo, coluna temporal, tipo de split, variáveis proibidas, idade, gates pendentes.

In [ ]:
if Path(SUPERVISED_CONFIG).exists():
    y = v.load_supervised_yaml(SUPERVISED_CONFIG)
    v.a3_config_audit(y, cfg, rep, df.attrs.get("schema_cols"))
    v.a3_forbidden(v.supervised_features(y), "supervisionado", cfg, rep)
else:
    print("supervised.yaml não encontrado:", SUPERVISED_CONFIG)

## 2. O desfecho não pode aparecer no clustering

In [ ]:
if FEATURES_CLUSTERING and Path(FEATURES_CLUSTERING).exists():
    v.a3_forbidden(v.load_feature_list(FEATURES_CLUSTERING), "clustering", cfg, rep)
else:
    print("Informe FEATURES_CLUSTERING (lista de features da matriz de clustering de B).")

## 3. Divisão temporal e proteção do holdout
Desenvolvimento 2022–2024 e holdout 2025 (conforme `configs/decisoes.yaml`).

In [ ]:
cohort = v.a1_cohort(df, cfg)
print(f"Coorte: {len(cohort):,} registros")
v.a3_temporal_split(cohort, cfg, rep)
display(pd.read_csv(Path(OUT) / "a3_split_temporal.csv"))

## 4. Varredura de vazamento
AUC univariada ≈ 1 contra o alvo é sinal de variável posterior ao desfecho (indicativo, não prova).

In [ ]:
v.a3_leak_scan(cohort, cfg, rep)
display(pd.read_csv(Path(OUT) / "a3_varredura_vazamento.csv").head(40))

## 5. Métricas excessivamente altas e auditoria dos logs de B e C

In [ ]:
if RESULTS and Path(RESULTS).exists():
    v.a3_suspicious_metrics(pd.read_csv(RESULTS), cfg, rep)
else:
    print("Sem tabela de resultados (RESULTS) ainda.")
v.a3_audit_logs(LOGS_DIR, cfg, rep)

## 6. Análises guiadas (manuais — registre a interpretação)
- [ ] Revisar lista final de features (desfecho fora do clustering)
- [ ] Confirmar divisão 2019–2024 / 2025 e métricas de clustering e de classificação (papel de cada uma)
- [ ] Executar uma rodada K-Means preparada e interpretar a tabela de perfis
- [ ] Executar a regressão logística preparada: coeficientes, probabilidades e métricas
- [ ] Comparar ROC-AUC e PR-AUC e explicar por que não são equivalentes
- [ ] Revisar uma curva de calibração (sub/superestimação)
- [ ] Investigar métricas altas demais como possível vazamento

Busque por: *silhouette score interpretation scikit-learn*, *precision recall curve imbalanced classification scikit-learn*, *probability calibration curve reliability diagram scikit-learn*, *TimeSeriesSplit scikit-learn chronological validation*.

## 7. Decisões e atas dos Gates 2 e 3

In [ ]:
E = "A3"
v.log_decision(OUT, "D-10", "Split temporal", f"Desenvolvimento {cfg['dev_years']} / holdout {cfg['holdout_years']} por DT_INTERNA", E,
               impacto="supervised.yaml: modo 'datas' e coluna_tempo = dt_interna (G2)")

DECISOES_G2 = ["(preencher) decisões do Gate 2 — desenho experimental"]
PENDENCIAS_G2 = ["(preencher)"]
v.write_ata(OUT, "Ata Gate 2 (11/10)", "2026-10-11", DECISOES_G2, PENDENCIAS_G2, riscos=[], rep=rep)

DECISOES_G3 = ["(preencher) configurações inviáveis eliminadas"]
PENDENCIAS_G3 = ["(preencher)"]
v.write_ata(OUT, "Ata Gate 3 (14/10)", "2026-10-14", DECISOES_G3, PENDENCIAS_G3, riscos=[], rep=rep)
print(sorted(p.name for p in Path(OUT).glob("ata_*.md")))

## 8. Fechamento

In [ ]:
path = rep.write_md()
print("Relatório:", path)
display(pd.DataFrame(rep.findings, columns=["etapa", "verificação", "status", "detalhe"]))